# Task 5 - Source metadata ingestion into MongoDB

Spark reads only `cpg.source-metadata.v1`, parses an explicit schema, and writes replacement upserts keyed by `_id=file_id`. Its checkpoint is retained on a Docker volume.

In [1]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import mongo_snapshot

snapshot = mongo_snapshot('867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7')
print(json.dumps(snapshot, indent=2))
assert snapshot['documents'] == snapshot['distinct_files'] == 61
assert snapshot['document']['_id'] == '867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7'
print('PASS: MongoDB has one replacement-upserted document per file')

{
  "documents": 61,
  "distinct_files": 61,
  "document": {
    "_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
    "path": "optimum/version.py",
    "content_hash": "079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce",
    "node_counts": {
      "AST": 25,
      "SYNTHETIC": 4
    },
    "edge_counts": {
      "AST": 24,
      "CFG": 9,
      "DFG": 3
    },
    "processed_at": "2026-07-22T01:08:09.918576Z",
    "run_id": "fb81462f6af147b5b74e33a33e1efa72",
    "kafka_offset": 136
  },
  "other_documents_count": 60,
  "other_documents_digest": "8aa80ee9361d8a1e4caf27fc02ca9ed83677a9820e0b2dbe011d53cdaaea0303"
}
PASS: MongoDB has one replacement-upserted document per file


In [2]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import checkpoint_offset, kafka_end_offset

progress = {
    'spark_checkpoint_offset': checkpoint_offset(),
    'metadata_topic_end_offset': kafka_end_offset('cpg.source-metadata.v1'),
}
print(json.dumps(progress, indent=2))
assert progress['spark_checkpoint_offset'] == progress['metadata_topic_end_offset']
print('PASS: Spark checkpoint has consumed the metadata topic')

{
  "spark_checkpoint_offset": 138,
  "metadata_topic_end_offset": 138
}
PASS: Spark checkpoint has consumed the metadata topic


![MongoDB UI showing the upserted source metadata document](figures/mongodb-ui.png)

*MongoDB UI showing the upserted source metadata document*

## Reflection

The checkpoint tracks Kafka progress rather than file hashes. MongoDB document and distinct-ID counts prove that repeated metadata events replace rather than duplicate a file document.